In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
DRIVE_PATH = '/content/drive/MyDrive/CICIoT2023_Research'
print(sorted(os.listdir(DRIVE_PATH)))

['X_test_cat8.csv', 'X_train_balanced_8cat_smoteenn.csv', 'X_train_cat8.csv', 'X_val_cat8.csv', 'balanced_class_counts_8cat_smoteenn.json', 'cat8_mapping.json', 'category_encoding_8cat.json', 'feature_columns.json', 'legacy_old', 'results', 'split_summary.json', 'y_test_cat8.csv', 'y_train_balanced_8cat_smoteenn.csv', 'y_train_cat8.csv', 'y_val_cat8.csv']


In [ ]:
import pandas as pd
import numpy as np
import json
import os
import gc

from collections import Counter
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN

DRIVE_PATH = '/content/drive/MyDrive/CICIoT2023_Research'

CATEGORY_ENCODING = {
    'DDoS': 0,
    'DoS': 1,
    'Mirai': 2,
    'Benign': 3,
    'Recon': 4,
    'Spoofing': 5,
    'Web': 6,
    'BruteForce': 7
}

ID_TO_CATEGORY = {v: k for k, v in CATEGORY_ENCODING.items()}

print("CATEGORY_ENCODING:")
print(CATEGORY_ENCODING)

CATEGORY_ENCODING:
{'DDoS': 0, 'DoS': 1, 'Mirai': 2, 'Benign': 3, 'Recon': 4, 'Spoofing': 5, 'Web': 6, 'BruteForce': 7}


In [ ]:
files = os.listdir(DRIVE_PATH)
print(files)

['X_train_balanced_8cat_smoteenn.csv', 'y_train_balanced_8cat_smoteenn.csv', 'category_encoding_8cat.json', 'balanced_class_counts_8cat_smoteenn.json', 'results', 'X_train_cat8.csv', 'X_val_cat8.csv', 'X_test_cat8.csv', 'y_train_cat8.csv', 'y_val_cat8.csv', 'y_test_cat8.csv', 'feature_columns.json', 'cat8_mapping.json', 'split_summary.json', 'legacy_old']


In [ ]:
print("Loading y_train_cat8...")
y_train_cat = pd.read_csv(f'{DRIVE_PATH}/y_train_cat8.csv').squeeze()

print("Train category distribution:")
for k, v in y_train_cat.value_counts().sort_index().items():
    print(f"{k} ({ID_TO_CATEGORY[k]}): {v}")

print(f"\nNull labels: {y_train_cat.isna().sum()}")

Loading y_train_cat8...
Train category distribution:
0 (DDoS): 8604445
1 (DoS): 2925190
2 (Mirai): 1651428
3 (Benign): 733115
4 (Recon): 458825
5 (Spoofing): 305243
6 (Web): 16595
7 (BruteForce): 8764

Null labels: 0


In [ ]:
print("Loading X_train_cat8...")
X_train = pd.read_csv(f'{DRIVE_PATH}/X_train_cat8.csv')
print("X_train shape:", X_train.shape)

gc.collect()

Loading X_train_cat8...
X_train shape: (14703605, 39)


0

In [ ]:
UNDER_TARGETS = {
    CATEGORY_ENCODING['DDoS']: 500_000,
    CATEGORY_ENCODING['DoS']: 500_000,
    CATEGORY_ENCODING['Mirai']: 500_000,
    CATEGORY_ENCODING['Benign']: 500_000,
    CATEGORY_ENCODING['Recon']: 200_000,
    CATEGORY_ENCODING['Spoofing']: 200_000,
}

current_counts = Counter(y_train_cat)
sampling_strategy = {
    k: v for k, v in UNDER_TARGETS.items()
    if current_counts[k] > v
}

print("Original counts:")
print(dict(sorted(current_counts.items())))

print("\nApplying RandomUnderSampler...")
rus = RandomUnderSampler(
    sampling_strategy=sampling_strategy,
    random_state=42
)

X_under, y_under = rus.fit_resample(X_train, y_train_cat)

print("After undersampling:", X_under.shape)
print(dict(sorted(Counter(y_under).items())))

del X_train, y_train_cat
gc.collect()

Original counts:
{0: 8604445, 1: 2925190, 2: 1651428, 3: 733115, 4: 458825, 5: 305243, 6: 16595, 7: 8764}

Applying RandomUnderSampler...
After undersampling: (2425359, 39)
{0: 500000, 1: 500000, 2: 500000, 3: 500000, 4: 200000, 5: 200000, 6: 16595, 7: 8764}


0

In [ ]:
OVER_TARGETS = {
    CATEGORY_ENCODING['Web']: 100_000,
    CATEGORY_ENCODING['BruteForce']: 50_000,
}

print("Applying SMOTE-ENN...")
smote_enn = SMOTEENN(
    sampling_strategy=OVER_TARGETS,
    random_state=42,
    n_jobs=-1
)

X_bal, y_bal = smote_enn.fit_resample(X_under, y_under)

final_counts = Counter(y_bal)

print("After SMOTE-ENN:", X_bal.shape)
for k, v in sorted(final_counts.items()):
    print(f"{k} ({ID_TO_CATEGORY[k]}): {v}")

del X_under, y_under
gc.collect()

Applying SMOTE-ENN...


In [ ]:
print("Saving balanced cat8 training set...")

X_bal_df = pd.DataFrame(X_bal)
y_bal_ser = pd.Series(y_bal, name="category_label")

X_bal_df.to_csv(f'{DRIVE_PATH}/X_train_balanced_8cat_smoteenn.csv', index=False)
y_bal_ser.to_csv(f'{DRIVE_PATH}/y_train_balanced_8cat_smoteenn.csv', index=False)

with open(f'{DRIVE_PATH}/category_encoding_8cat.json', 'w') as f:
    json.dump(CATEGORY_ENCODING, f, indent=2)

final_counts_json = {str(k): int(v) for k, v in sorted(Counter(y_bal).items())}
with open(f'{DRIVE_PATH}/balanced_class_counts_8cat_smoteenn.json', 'w') as f:
    json.dump(final_counts_json, f, indent=2)

balancing_summary = {
    "input_feature_file": "X_train_cat8.csv",
    "input_label_file": "y_train_cat8.csv",
    "undersampling_targets": {str(k): int(v) for k, v in UNDER_TARGETS.items()},
    "smoteenn_targets": {str(k): int(v) for k, v in OVER_TARGETS.items()},
    "final_counts": final_counts_json,
    "final_shape_X": list(X_bal_df.shape),
    "final_shape_y": int(len(y_bal_ser))
}

with open(f'{DRIVE_PATH}/balancing_summary_8cat_smoteenn.json', 'w') as f:
    json.dump(balancing_summary, f, indent=2)

print("Saved successfully.")